# PainCircuitProfiler — Example Analysis

This notebook demonstrates the full analysis workflow:
integrating Allen Brain Atlas **connectivity data** with **ISH gene expression** to
identify hub regions in the mouse pain circuit.

## Scientific question
> *Which brain regions in the ascending pain pathway are best positioned to influence
> opioid signalling — i.e., they project strongly to opioid-receptor-rich areas
> AND are highly interconnected within the pain network?*

## Dataset
- **Connectivity**: Allen Mouse Brain Connectivity Atlas (Oh et al., *Nature* 2014)
- **Expression**: Allen Mouse Brain Atlas ISH (Lein et al., *Nature* 2007)
- **Regions**: 18 pain-related regions spanning cortex → amygdala → thalamus → midbrain → hindbrain
- **Genes**: Oprm1, Oprd1, Oprk1 (opioid receptors); Penk, Pdyn, Pomc (endogenous opioids)

---
**Note:** First run downloads Allen Atlas data (~100 MB). Subsequent runs use local cache and are fast.

## 0. Setup

In [ ]:
import sys
sys.path.insert(0, '..')   # Add project root to path when running from notebooks/

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import matplotlib.pyplot as plt

# PainCircuitProfiler modules
from pain_circuit_profiler.config import PAIN_REGIONS, GENES, ANALYSIS_DEFAULTS
from pain_circuit_profiler.data.connectivity    import ConnectivityLoader
from pain_circuit_profiler.data.gene_expression import GeneExpressionLoader
from pain_circuit_profiler.analysis.hub_metrics import run_analysis, print_top_hubs
from pain_circuit_profiler.visualization.plots  import (
    plot_expression_heatmap,
    plot_connectivity_heatmap,
    plot_network,
    plot_hub_scores,
    plot_integrated_dashboard,
)

# ---- Configuration ----
CACHE_DIR  = '../allen_connectivity_cache'
OUTPUT_DIR = '../outputs'

print(f'Pain regions defined: {len(PAIN_REGIONS)}')
print(f'Target genes defined: {len(GENES)}')

## 1. Inspect the region and gene definitions

Before loading any data, let's see exactly which regions and genes we're working with.
All of this is editable in `pain_circuit_profiler/config.py`.

In [ ]:
# View the region table
region_rows = [
    {'Acronym': acr, 'Label': info['label'],
     'Category': info['category'], 'Role': info['description']}
    for acr, info in PAIN_REGIONS.items()
]
pd.DataFrame(region_rows).set_index('Acronym')

In [ ]:
# View the gene table
gene_rows = [
    {'Gene': g, 'Full name': info['name'],
     'Pathway': info['pathway'], 'Function': info['function']}
    for g, info in GENES.items()
]
pd.DataFrame(gene_rows).set_index('Gene')

## 2. Load connectivity data

This step:
1. Downloads the Allen structure tree (brain region hierarchy)
2. Looks up the numeric Allen ID for each of our acronyms
3. For each source region, retrieves all injection experiments from the Connectivity Atlas
4. Queries projection energy into all other pain regions
5. Averages across experiments (biological replicates) → returns an N×N matrix

**Expected time:** 2–5 min on first run; <30 sec on subsequent runs (data is cached locally).

In [ ]:
conn_loader = ConnectivityLoader(cache_dir=CACHE_DIR, resolution=100)

connectivity, conn_metadata = conn_loader.load(regions=PAIN_REGIONS)

print(f'Connectivity matrix: {connectivity.shape[0]} regions × {connectivity.shape[1]} regions')
print(f'\nExperiments per injection region:')
for acr, meta in conn_metadata.items():
    print(f'  {meta["label"]:<28}  {meta["n_experiments"]:>3} experiments')

In [ ]:
# Preview the connectivity matrix
# Rows = source (injection site), Columns = target (projection received)
# Values = mean projection_energy across experiments
connectivity.round(4)

## 3. Load gene expression data

This step queries the Allen Mouse Brain Atlas ISH data via the RMA REST API.
For each gene, it:
1. Finds all ISH section data sets for that gene in adult mouse brain
2. Queries `expression_energy` per structure for our pain regions
3. Averages across data sets → one value per (gene, region) pair

**Expected time:** 1–3 min on first run; near-instant on subsequent runs (cached as JSON).

In [ ]:
# Get the Allen structure IDs we need (connectivity loader already has these)
acronym_to_id = conn_loader.get_structure_ids(list(PAIN_REGIONS.keys()))
id_to_label   = {v: PAIN_REGIONS[k]['label'] for k, v in acronym_to_id.items()}
structure_ids = list(acronym_to_id.values())

gene_list = list(GENES.keys())

expr_loader = GeneExpressionLoader(cache_dir=CACHE_DIR)
expression  = expr_loader.load(genes=gene_list, structure_ids=structure_ids)

# Translate Allen IDs → display labels for readability
expression.index = [id_to_label.get(i, str(i)) for i in expression.index]

print(f'Expression matrix: {expression.shape[0]} regions × {expression.shape[1]} genes')
expression.round(4)

## 4. Run hub analysis

The hub scoring pipeline (defined in `analysis/hub_metrics.py`) integrates both datasets:

1. **Expression hubs** — regions in the top 25% of expression for a gene
2. **Projection scores** — for each region: sum of projections *to* expression hubs
3. **Hub candidates** — regions in the top 50% of projection scores
4. **Interconnectivity** — among candidates, bidirectional connectivity within the group
5. **Hub score** — geometric mean of (normalised projection) × (normalised interconnectivity)

A high hub score = *projects strongly to opioid-rich areas* AND *is highly embedded in the network*.

In [ ]:
hub_results = run_analysis(
    connectivity,
    expression,
    id_to_label           = None,    # Already translated above
    expression_percentile = 75,      # Top 25% expression = hub
    projection_percentile = 50,      # Top 50% projection = candidate
)

print_top_hubs(hub_results, top_n=10)

In [ ]:
# Detailed hub scores for Oprm1 (mu-opioid receptor)
hub_results['per_gene']['Oprm1'].round(4)

## 5. Visualize — Gene expression heatmap

Shows how strongly each gene is expressed in each pain region.
Colour intensity = expression energy from ISH.
Region labels are colour-coded by anatomical category.

In [ ]:
fig = plot_expression_heatmap(
    expression,
    output_path=f'{OUTPUT_DIR}/01_expression_heatmap.png'
)
plt.show()

## 6. Visualize — Connectivity heatmap

Shows axonal projection strength between all pairs of pain regions.
Values are log₁₀-transformed (projection energies span orders of magnitude).
Read as: *rows project TO columns*.

In [ ]:
fig = plot_connectivity_heatmap(
    connectivity,
    output_path=f'{OUTPUT_DIR}/02_connectivity_heatmap.png'
)
plt.show()

## 7. Visualize — Network diagram

Directed graph where:
- **Node colour** = Oprm1 expression energy (pale → deep red)
- **Node size** = out-degree (number of significant projections sent)
- **Edge width** = projection strength
- **Arrow direction** = anatomical projection direction

Nodes that are both large (many projections) and dark (Oprm1-rich) are
primary candidates for opioid circuit modulation.

In [ ]:
fig = plot_network(
    connectivity,
    expression_series = expression['Oprm1'],
    gene_name         = 'Oprm1',
    edge_threshold    = 0.001,
    layout            = 'spring',
    output_path       = f'{OUTPUT_DIR}/03_network_Oprm1.png'
)
plt.show()

In [ ]:
# Try a different gene — prodynorphin (kappa system)
fig = plot_network(
    connectivity,
    expression_series = expression['Pdyn'],
    gene_name         = 'Pdyn',
    edge_threshold    = 0.001,
)
plt.show()

## 8. Visualize — Hub score rankings

One panel per gene, showing top-ranked regions by hub score.
★ marks regions classified as expression hubs for that gene.

In [ ]:
fig = plot_hub_scores(
    hub_results['per_gene'],
    top_n       = 10,
    output_path = f'{OUTPUT_DIR}/04_hub_scores.png'
)
plt.show()

## 9. Visualize — Integrated dashboard

Four-panel summary figure combining all views.
Suitable for a poster, paper supplement, or README figure.

In [ ]:
fig = plot_integrated_dashboard(
    connectivity,
    expression,
    hub_results,
    gene        = 'Oprm1',
    output_path = f'{OUTPUT_DIR}/05_integrated_dashboard.png'
)
plt.show()

## 10. Extending the analysis

### A. Restrict to a specific hemisphere
```python
connectivity_ipsi, _ = conn_loader.load(
    regions=PAIN_REGIONS,
    hemisphere_ids=[2]   # 1=left, 2=right, 3=bilateral
)
```

### B. Add custom regions
```python
from pain_circuit_profiler.config import PAIN_REGIONS

my_regions = dict(PAIN_REGIONS)   # start from defaults
my_regions['NTS'] = {
    'label':       'NTS (medulla)',
    'category':    'Hindbrain',
    'description': 'Nucleus tractus solitarius — visceral pain relay'
}
my_regions['RVM'] = {
    'label':       'RVM',
    'category':    'Hindbrain',
    'description': 'Rostral ventromedial medulla — descending modulation'
}
connectivity, _ = conn_loader.load(regions=my_regions)
```

### C. Query non-opioid genes
```python
extra_genes = ['Grm5', 'Cnr1', 'Trpv1']   # mGluR5, CB1, TRPV1
expression2 = expr_loader.load(
    genes=extra_genes,
    structure_ids=structure_ids
)
```

### D. Integrate your own experimental data
If you have TRAP2 activity data or patch-seq expression profiles, you can replace
the Allen ISH expression matrix with your own DataFrame (same format: index=region labels,
columns=genes/conditions) and pass it directly to `run_analysis()`:
```python
# your_data.csv: rows=regions, columns=genes/conditions
my_expr = pd.read_csv('your_data.csv', index_col=0)
hub_results = run_analysis(connectivity, my_expr)
```

---
## Data sources

- Oh, S.W. et al. (2014). A mesoscale connectome of the mouse brain. *Nature*, 508, 207–214.
- Lein, E.S. et al. (2007). Genome-wide atlas of gene expression in the adult mouse brain. *Nature*, 445, 168–176.
- Allen Mouse Brain Atlas: https://mouse.brain-map.org
- Allen Mouse Connectivity Atlas: https://connectivity.brain-map.org